In [ ]:
!pip install --quiet transformers datasets torch

In [ ]:
import random
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import pandas as pd
from tqdm import tqdm

# Seed
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
# Use small GPT-2 for fast training/generation
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# Add padding token for generation
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = model.config.eos_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Using device: {device}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Using device: cpu


In [ ]:
spam_prompts = [
    "Congratulations! You won a free lottery of",
    "You have been selected for a cash prize of",
    "Earn money fast by",
    "Get your free gift card now at",
    "Claim your reward by clicking",
]

ham_prompts = [
    "Dear John, I wanted to update you about",
    "Meeting scheduled for tomorrow at",
    "Here is the report for last month",
    "Can we reschedule our appointment to",
    "Please review the attached document",
]


In [ ]:
# Parameters
n_samples_per_class = 50  # smaller for testing, scale up later
max_length = 50
batch_size = 5  # generate 5 emails per prompt at once

emails = []

def generate_emails_batch(prompts, label):
    for prompt in tqdm(prompts):
        input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
        attention_mask = torch.ones_like(input_ids).to(device)

        # Generate multiple sequences in a single forward pass
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=max_length,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            num_return_sequences=batch_size,
            pad_token_id=tokenizer.eos_token_id
        )

        # Collect generated texts
        for i, o in enumerate(output):
            if i >= n_samples_per_class // len(prompts):
                break
            text = tokenizer.decode(o, skip_special_tokens=True)
            emails.append({"text": text, "label": label})

# Generate spam emails
generate_emails_batch(spam_prompts, label=1)
# Generate non-spam emails
generate_emails_batch(ham_prompts, label=0)

# Create DataFrame
df = pd.DataFrame(emails)
print(df.head())
print("\nClass distribution:\n", df['label'].value_counts())


100%|██████████| 5/5 [00:40<00:00,  8.02s/it]

                                                text  label
0  Congratulations! You won a free lottery of the...      1
1  Congratulations! You won a free lottery of all...      1
2  Congratulations! You won a free lottery of the...      1
3  Congratulations! You won a free lottery of any...      1
4  Congratulations! You won a free lottery of $3,...      1

Class distribution:
 label
1    25
0    25
Name: count, dtype: int64


In [ ]:
# Save dataset in Data folder
import os

os.makedirs("../Data", exist_ok=True)   # make Data folder if not exists
df.to_csv("../Data/synthetic_email_dataset.csv", index=False)

print("Dataset saved at ../Data/synthetic_email_dataset.csv")



Dataset saved at ../Data/synthetic_email_dataset.csv
